
# Deepfake Detection Challenge Baseline


- **ImageNet pretrained backbone 1개**
- **GAN 기반 얼굴 데이터셋 1개** 사용
- 최종적으로 `submission_baseline.csv` 생성

---

## 목차

1. 환경 설정  
2. 경로 및 하이퍼파라미터 설정  
3. 대회 데이터 압축 해제  
4. 학습 데이터 다운로드 및 로드  
5. 학습/검증 데이터셋 구성  
6. 모델 정의  
7. 학습  
8. 검증  
9. 테스트 추론 및 제출 파일 생성  

---

## 설계

- backbone: `torchvision`의 `ResNet18`
- backbone 대부분 **freeze**
- 외부 데이터는 **GAN 기반 얼굴 데이터셋 1개만** 사용
- 학습 데이터는 전체를 다 쓰지 않고 **작은 subset만 사용**




## 학습 데이터셋

이 베이스라인은 **Kaggle의 `xhlulu/140k-real-and-fake-faces`** 데이터셋을 사용합니다.

- 총 **140,000장**
- **real 70,000장 / fake 70,000장**
- fake 이미지는 **StyleGAN 기반 GAN 얼굴**로 구성되어 있습니다.

이 노트북에서는 라벨을 다음과 같이 사용합니다.

- **real = 0**
- **fake = 1**

즉, `real/` 폴더 이미지는 모두 `0`, `fake/` 폴더 이미지는 모두 `1`로 사용합니다.


In [1]:
# ============================================================
# 1. 필수 패키지 설치
# ============================================================

!pip cache purge
!pip install --user kagglehub pillow==11.1.0 --ignore-installed
!pip install timm


Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ============================================================
# 2. 라이브러리 / 경로 / 하이퍼파라미터 설정
# ============================================================

import os
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import transforms, datasets

import kagglehub
import timm
ImageFile.LOAD_TRUNCATED_IMAGES = True

# -------------------------------
# 재현성
# -------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -------------------------------
# 경로
# -------------------------------

WORKING_DIR = Path(os.getcwd())

COMP_DIR = "/kaggle/input/datasets/seoyeon056/competition"
TEST_DIR = os.path.join(COMP_DIR, "test")

TRAIN_DIR = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
DATASET_SLUG = "xhlulu/140k-real-and-fake-faces"
# -------------------------------
# 학습 설정
# -------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 256
BATCH_SIZE = 32
NUM_WORKERS = 2

TRAIN_PER_CLASS = 8000
VAL_PER_CLASS = 2000

EPOCHS = 5
LR = 1e-4

print("DEVICE:", DEVICE)


DEVICE: cuda


In [3]:
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level <= 2:
        print(root, f"({len(files)} files)")

/kaggle/input (0 files)
/kaggle/input/datasets (0 files)
/kaggle/input/datasets/greatgamedota (0 files)
/kaggle/input/datasets/xhlulu (0 files)
/kaggle/input/datasets/seoyeon056 (0 files)


In [4]:
# ============================================================
# 4. 테스트 데이터와 제출 형식 불러오기
# ============================================================
TEST_CSV_PATH = os.path.join(COMP_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(COMP_DIR, "sample_submission.csv")

assert os.path.exists(TEST_DIR), "test 폴더가 없습니다."
assert os.path.exists(TEST_CSV_PATH), "test.csv가 없습니다."
assert os.path.exists(SAMPLE_SUB_PATH), "sample_submission.csv가 없습니다."

test_df = pd.read_csv(TEST_CSV_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(test_df.head())
print(sample_sub.head())
print("테스트 이미지 수:", len(test_df))

        id         file_path
0  test001  test/test001.png
1  test002  test/test002.png
2  test003  test/test003.png
3  test004  test/test004.png
4  test005  test/test005.png
        id  score
0  test001    0.5
1  test002    0.5
2  test003    0.5
3  test004    0.5
4  test005    0.5
테스트 이미지 수: 300



## Kaggle 데이터셋 다운로드

이 셀은 **GAN 기반 얼굴 데이터셋**을 내려받습니다.  
Colab에서 처음 실행하면 Kaggle 인증이 필요할 수 있습니다.


In [5]:
# ============================================================
# 5. 학습 데이터 다운로드
# ============================================================
# xhlulu/140k-real-and-fake-faces
# - real/ 70k
# - fake/ 70k (StyleGAN 계열)
DATASET_DIR = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Downloaded dataset to:", DATASET_DIR)

# 데이터셋 내부 구조 확인
for p in DATASET_DIR.iterdir():
    print(p.name)

Downloaded dataset to: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces
valid.csv
real_vs_fake
train.csv
test.csv


In [6]:
# ============================================================
# 6. 이미지 전처리 정의
# ============================================================
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale = (0.8,1.0)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [7]:
# ============================================================
# 7. 학습 데이터셋 구성
#    ImageFolder 구조를 사용합니다.
#    기대 폴더:
#      real/
#      fake/
# ============================================================
# 데이터셋 루트 자동 탐색
candidate_roots = [DATASET_DIR]
for sub in DATASET_DIR.rglob("*"):
    if sub.is_dir():
        child_names = sorted([p.name.lower() for p in sub.iterdir() if p.is_dir()])
        if "real" in child_names and "fake" in child_names:
            candidate_roots.append(sub)

dataset_root = None
for c in candidate_roots:
    child_names = sorted([p.name.lower() for p in c.iterdir() if p.is_dir()])
    if "real" in child_names and "fake" in child_names:
        dataset_root = c
        break

assert dataset_root is not None, "real/ fake/ 폴더 구조를 찾지 못했습니다."
print("Using dataset root:", dataset_root)

full_train_ds = datasets.ImageFolder(root=str(dataset_root), transform=train_tfms)
full_valid_ds = datasets.ImageFolder(root=str(dataset_root), transform=valid_tfms)

print("class_to_idx:", full_train_ds.class_to_idx)

Using dataset root: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid
class_to_idx: {'fake': 0, 'real': 1}


In [8]:
# ============================================================
# 8. 라벨 매핑 정리
#    최종 학습 라벨은 real=0, fake=1 로 맞춥니다.
# ============================================================
class_to_idx = full_train_ds.class_to_idx

def original_to_target_label(orig_label: int):
    # ImageFolder의 class 인덱스를 우리 target label로 변환
    # fake -> 1, real -> 0
    class_name = [k for k, v in class_to_idx.items() if v == orig_label][0].lower()
    if class_name == "fake":
        return 1
    elif class_name == "real":
        return 0
    else:
        raise ValueError(f"Unknown class name: {class_name}")

# 클래스별 인덱스 모으기
real_indices = []
fake_indices = []

for i, (_, y) in enumerate(full_train_ds.samples):
    class_name = [k for k, v in class_to_idx.items() if v == y][0].lower()
    if class_name == "real":
        real_indices.append(i)
    elif class_name == "fake":
        fake_indices.append(i)

random.Random(SEED).shuffle(real_indices)
random.Random(SEED).shuffle(fake_indices)

train_real_idx = real_indices[:TRAIN_PER_CLASS]
val_real_idx = real_indices[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS]

train_fake_idx = fake_indices[:TRAIN_PER_CLASS]
val_fake_idx = fake_indices[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS]

print("train real:", len(train_real_idx))
print("train fake:", len(train_fake_idx))
print("val real:", len(val_real_idx))
print("val fake:", len(val_fake_idx))

train real: 8000
train fake: 8000
val real: 2000
val fake: 2000


In [9]:
# ============================================================
# 9. 라벨 변환용 Wrapper Dataset
# ============================================================
class RemapLabelSubset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base_dataset = base_dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, orig_y = self.base_dataset[self.indices[idx]]
        y = torch.tensor(float(original_to_target_label(int(orig_y))), dtype=torch.float32)
        return x, y

train_dataset = torch.utils.data.ConcatDataset([
    RemapLabelSubset(full_train_ds, train_real_idx),
    RemapLabelSubset(full_train_ds, train_fake_idx),
])

val_dataset = torch.utils.data.ConcatDataset([
    RemapLabelSubset(full_valid_ds, val_real_idx),
    RemapLabelSubset(full_valid_ds, val_fake_idx),
])

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Train size: 16000
Val size: 4000


In [10]:
# ============================================================
# 10. DataLoader
# ============================================================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [11]:
# ============================================================
# 11. 모델 정의
# - ResNet18 backbone
# - backbone freeze
# - 마지막 fc만 학습
# ============================================================
model = timm.create_model('efficientnet_b3', pretrained = True, num_classes = 1)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(model)

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

EfficientNet(
  (conv_stem): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
        (bn1): BatchNormAct2d(
          40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(40, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (b

In [12]:
# ============================================================
# 12. ROC-AUC 계산 함수
# ============================================================
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def evaluate_auc(model, loader):
    model.eval()
    all_probs = []
    all_labels = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x).squeeze(1)
        probs = torch.sigmoid(logits)

        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    auc = roc_auc_score(all_labels, all_probs)
    return auc

In [13]:
# ============================================================
# 13. 학습
# ============================================================
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_loader.dataset)
    val_auc = evaluate_auc(model, val_loader)

    print(f"[Epoch {epoch+1}] train_loss={train_loss:.4f} | val_auc={val_auc:.4f}")

import gc
gc.collect()
torch.cuda.empty_cache()

Epoch 1/5:   0%|          | 0/500 [00:00<?, ?it/s]

[Epoch 1] train_loss=0.5487 | val_auc=0.9793


Epoch 2/5:   0%|          | 0/500 [00:00<?, ?it/s]

[Epoch 2] train_loss=0.1649 | val_auc=0.9919


Epoch 3/5:   0%|          | 0/500 [00:00<?, ?it/s]

[Epoch 3] train_loss=0.0892 | val_auc=0.9957


Epoch 4/5:   0%|          | 0/500 [00:00<?, ?it/s]

[Epoch 4] train_loss=0.0517 | val_auc=0.9961


Epoch 5/5:   0%|          | 0/500 [00:00<?, ?it/s]

[Epoch 5] train_loss=0.0449 | val_auc=0.9967


In [14]:
# ============================================================
# 14. 대회 테스트셋 Dataset
# ============================================================
class CompetitionTestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root_dir / f"{row['id']}.png"
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return row["id"], img

test_dataset = CompetitionTestDataset(test_df, TEST_DIR, transform=valid_tfms)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [15]:
# ============================================================
# 15. 테스트셋 추론
# ============================================================
@torch.no_grad()
def predict_test(model, loader):
    model.eval()
    ids = []
    probs = []

    for batch_ids, x in tqdm(loader, desc="Predicting test set"):
        x = x.to(DEVICE)
        logits = model(x).squeeze(1)
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy().tolist()

        ids.extend(list(batch_ids))
        probs.extend(batch_probs)

    return ids, probs

pred_ids, pred_scores = predict_test(model, test_loader)

Predicting test set:   0%|          | 0/10 [00:00<?, ?it/s]

In [16]:
# ============================================================
# 16. 제출 파일 생성
# ============================================================
submission = pd.DataFrame({
    "id": pred_ids,
    "score": pred_scores
})

submission = sample_sub[["id"]].merge(submission, on="id", how="left")

submission_path = "submission.csv"
submission["score"] = 1 - submission["score"] 
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
submission.head()

Saved submission to: submission.csv


,id,score
0,test001,0.999970
1,test002,0.999974
2,test003,0.999909
3,test004,0.999999
4,test005,1.000000


In [17]:
print(submission["score"].describe())
print(submission["score"].value_counts(bins=5))

count    300.000000
mean       0.994848
std        0.049336
min        0.318062
25%        0.999967
50%        0.999999
75%        1.000000
max        1.000000
Name: score, dtype: float64
(0.864, 1.0]      297
(0.316, 0.454]      1
(0.454, 0.591]      1
(0.727, 0.864]      1
(0.591, 0.727]      0
Name: count, dtype: int64


In [18]:
submission = pd.DataFrame({
    "id": pred_ids,
    "score": pred_scores
})

submission = sample_sub[["id"]].merge(submission, on="id", how="left")

import numpy as np
temperature = 5.0
logits = submission["score"].apply(lambda p: np.log(p / (1 - p + 1e-7))).values
submission["score"] = 1 / (1 + np.exp(-logits / temperature))
submission["score"] = 1 - submission["score"]  # 반전

print(submission["score"].describe())

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved submission to:", submission_path)
submission.head()

count    300.000000
mean       0.915328
std        0.080303
min        0.461939
25%        0.887417
50%        0.942107
75%        0.969143
max        0.997808
Name: score, dtype: float64
Saved submission to: submission.csv


,id,score
0,test001,0.889172
1,test002,0.892123
2,test003,0.865424
3,test004,0.945961
4,test005,0.970790


In [19]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level <= 2:
        print(root, f"({len(files)} files)")

/kaggle/input (0 files)
/kaggle/input/datasets (0 files)
/kaggle/input/datasets/greatgamedota (0 files)
/kaggle/input/datasets/xhlulu (0 files)
/kaggle/input/datasets/seoyeon056 (0 files)
